# Решения: кластеры и аномалии

**Для преподавателя.** Полный эталон к `lesson.ipynb` и `homework.ipynb`; ученикам до сдачи не показывать.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def find_orders_csv() -> Path:
    for path in (Path("orders_slim.csv"), Path("../../data/orders_slim.csv")):
        if path.exists():
            return path.resolve()
    raise FileNotFoundError(
        "orders_slim.csv не найден рядом с ноутбуком или в ../../data/"
    )


CSV_PATH = find_orders_csv()
DATE_COLUMNS = [
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
    "order_delivered_customer_date",
]
df = pd.read_csv(CSV_PATH, parse_dates=DATE_COLUMNS)
assert len(df) > 0
assert df["order_id"].notna().all()
print(f"Загружено заказов: {len(df)}")

from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import StandardScaler

FEATURES = ["delivery_days", "freight_value", "delay_days"]


## Урок. 1. Матрица

In [ ]:
X = df[FEATURES].copy()
Xs = StandardScaler().fit_transform(X)
assert "is_late" not in X and Xs.shape == X.shape


## Урок. 2–4. Две кластеризации

In [ ]:
labels_km = KMeans(n_clusters=min(3, len(df)), random_state=54, n_init=10).fit_predict(Xs)
labels_db = DBSCAN(eps=0.9, min_samples=4).fit_predict(Xs)
clustered = df.copy()
clustered["cluster_km"] = labels_km
clustered["cluster_db"] = labels_db
assert len(clustered) == len(df)


## Урок. 5. Профили

In [ ]:
profiles = clustered.groupby("cluster_km").agg(size=("order_id", "size"), delivery_days_mean=("delivery_days", "mean"), freight_median=("freight_value", "median"), delay_mean=("delay_days", "mean"), late_rate=("is_late", "mean")).round(2)
assert int(profiles["size"].sum()) == len(df)
print(profiles)


## Урок. 6–7. Кандидаты

In [ ]:
noise_candidates = clustered[clustered["cluster_db"].eq(-1)].copy()
top_delay_idx = clustered.nlargest(min(5, len(df)), "delay_days").index.tolist()
candidate_idx = set(noise_candidates.index) | set(top_delay_idx)
assert set(top_delay_idx) <= candidate_idx


## Урок. 8. Severity

In [ ]:
z = pd.DataFrame(Xs, columns=FEATURES, index=df.index)
severity = 0.7 * z["delay_days"] + 0.3 * z["freight_value"]
anomalies = clustered.loc[sorted(candidate_idx)].copy()
anomalies["severity"] = severity.loc[anomalies.index]
anomalies = anomalies.sort_values("severity", ascending=False)
assert anomalies["severity"].is_monotonic_decreasing


## Урок. 9. Причины

In [ ]:
reasons = []
for row in anomalies.itertuples():
    source = "DBSCAN пометил как шум" if row.cluster_db == -1 else "входит в top-delay"
    reasons.append(f"{source}; delay={row.delay_days} дней, freight={row.freight_value:.1f}.")
anomalies = anomalies.assign(reason=reasons)
assert all(len(text) >= 35 for text in reasons)


## Урок. 10. Записка хабу

In [ ]:
OPS_NOTE = f"Получено {clustered['cluster_km'].nunique()} KMeans-сегмента и {len(anomalies)} кандидатов для ручной проверки. Сначала проверить верхние строки severity: сочетание задержки и стоимости доставки. is_late использован только для описания профилей, не как признак. Список не доказывает ошибку: DBSCAN чувствителен к eps, а высокий delay может иметь штатную причину."
assert len(OPS_NOTE) >= 250 and str(len(anomalies)) in OPS_NOTE


## ДЗ. A1. Pipeline

In [ ]:
def cluster_orders(frame, eps=0.9):
    values = StandardScaler().fit_transform(frame[FEATURES])
    result = frame.copy()
    result["cluster_km"] = KMeans(n_clusters=min(3, len(frame)), random_state=54, n_init=10).fit_predict(values)
    result["cluster_db"] = DBSCAN(eps=eps, min_samples=4).fit_predict(values)
    return result

result = cluster_orders(df)
assert len(result) == len(df)


## ДЗ. A2. Таблица аномалий

In [ ]:
top_idx = set(result.nlargest(min(8, len(result)), "delay_days").index)
noise_idx = set(result.index[result["cluster_db"].eq(-1)])
chosen = list(top_idx | noise_idx)
anomaly_table = result.loc[chosen].sort_values(["delay_days", "freight_value"], ascending=False).head(min(12, len(result)))
assert 1 <= len(anomaly_table) <= min(12, len(df))


## ДЗ. A3. Quality gate

In [ ]:
checks = {"rows_preserved": len(result) == len(df), "three_features": len(FEATURES) == 3, "late_not_feature": "is_late" not in FEATURES, "ranked": anomaly_table["delay_days"].is_monotonic_decreasing}
assert set(checks.values()) == {True}


## ДЗ. Challenge

In [ ]:
def anomaly_ids(frame, eps):
    result = cluster_orders(frame, eps)
    ids = set(result.loc[result["cluster_db"].eq(-1), "order_id"])
    ids.update(result.nlargest(min(5, len(result)), "delay_days")["order_id"])
    return ids

ids_07, ids_11 = anomaly_ids(df, 0.7), anomaly_ids(df, 1.1)
union = ids_07 | ids_11
jaccard = len(ids_07 & ids_11) / len(union) if union else 1.0
EXECUTIVE_NOTE = (
    f"Кластерный анализ описывает типичные режимы доставки, а список аномалий выделяет заказы для ручной проверки. "
    f"При eps 0.7 и 1.1 сходство списков равно {jaccard:.2f}, поэтому состав зависит от настройки DBSCAN. "
    "Операционное действие: проверить первые заказы по задержке и стоимости, затем связать их с маршрутами. "
    "Ограничение: кластер не является причиной задержки, а аномалия не означает ошибку; is_late служит только описанием."
)
assert 0 <= jaccard <= 1 and len(EXECUTIVE_NOTE) >= 300
